# Continuous Brain v0.11.0 — Complete A100 Colab Runner

This notebook is designed for the current v0.11.0 lineage and is **resume-aware**.

It uses the Colab machine where it is safe to do so:

- **A100 CUDA** for model compute.
- **local `/content` SSD** for active project/data/checkpoint I/O.
- **Linux RAM page cache** to keep the small multi-GB nursery working set hot in the ~83 GB host RAM.
- **Google Drive** only for immutable inputs, durable checkpoints, and results.

It deliberately does **not** inflate batch sizes, enable reduced precision, or touch the reserved generalized blind seeds merely to fill 40 GB of VRAM. Those changes would alter the predeclared experiment. The 53.68M-parameter model simply does not require 40 GB for the canonical Phase A/gates.

### Current lineage anchors

- Bundle SHA-256: `be3bdfe94beb0561908e9f921170f32fe0a48e700d648c30931ee7f1bd0bda3e`
- Exact v0.11.0 expansion checkpoint: `086932388a9a410af46fea8e87cc63dd4b60161e03e247bf81fc88d28cfd577f`
- Your completed Phase A checkpoint: `b03a932a1f141c46213363fb990a722ae4e93dd57a20ffef25874899cab79e9a`
- Target routed steps: `6000`
- Future generalized blind seeds `51001–51100` are **never touched** by this notebook.

If your Drive state is already at 6000 routed steps, the training section will **skip automatically** and proceed to qualification.

In [ ]:
# Configuration — normally you should not need to change this cell.
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/continuous_brain_v110")
BUNDLE = DRIVE_ROOT / "continuous_brain_v110_colab_bundle.tar.gz"

EXPECTED_BUNDLE_SHA = "be3bdfe94beb0561908e9f921170f32fe0a48e700d648c30931ee7f1bd0bda3e"
EXPECTED_EXACT_EXPANSION_SHA = "086932388a9a410af46fea8e87cc63dd4b60161e03e247bf81fc88d28cfd577f"
EXPECTED_PHASE_A_SHA = "b03a932a1f141c46213363fb990a722ae4e93dd57a20ffef25874899cab79e9a"

EXPECTED_V104_SEAL_SHA = "8f00579cad8e283fac4103da250b1c768b5ad92353d628f8bc50700db4d63a4b"
EXPECTED_FORAGING_HOLDOUT_DIGEST = "1797991caa73463ebbed25f35e5240dba339f35e007b64a574195672bf2b2b5e"
EXPECTED_DETOUR_HOLDOUT_DIGEST = "a0ba1cb474601031626214b371e988afd187f345a0836a071056cc5764062551"

LOCAL_ROOT = Path("/content/cb_v110")
PROJECT = LOCAL_ROOT / "continuous_brain_project"
PAYLOAD = LOCAL_ROOT / "payload"
DATA = PAYLOAD / "datasets"

WORK = Path("/content/cb_v110_work")
LOCAL_STATE = WORK / "state" / "child_v110_growth"
LOCAL_RESULTS = WORK / "results"

DRIVE_STATE = DRIVE_ROOT / "state" / "child_v110_growth"
DRIVE_RESULTS = DRIVE_ROOT / "results"

WORLD_ASSETS = DRIVE_ROOT / "v110_world_retention_assets.tar.gz"
WORLD_ASSETS_SHA = DRIVE_ROOT / "v110_world_retention_assets.tar.gz.sha256"

TARGET_ROUTE_STEPS = 6000
RUN_PHASE_A_IF_NEEDED = True
WARM_DATASETS_IN_RAM_CACHE = True

# Formal gates remain canonical and single-process.
# This favors reproducibility over artificially filling the A100.
FORMAL_RANDOMIZED_DEV_SEED_START = 41001
FORMAL_RANDOMIZED_DEV_EPISODES = 100
P4B_REGRESSION_SEED_START = 12001
P4B_REGRESSION_EPISODES = 100

print("Configuration loaded.")

In [ ]:
# Mount Drive and inventory the Colab machine.
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, shutil, subprocess, torch, hashlib, json, platform, time

def gib(n): return n / 1024**3
def sha256(path, chunk=8*1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

cpu_count = os.cpu_count() or 1
meminfo = {}
with open("/proc/meminfo") as f:
    for line in f:
        k,v,*_ = line.split()
        meminfo[k.rstrip(":")] = int(v)*1024
disk = shutil.disk_usage("/content")

print("=== HOST ===")
print("CPU logical cores:", cpu_count)
print("System RAM total GiB:", round(gib(meminfo["MemTotal"]),2))
print("System RAM available GiB:", round(gib(meminfo["MemAvailable"]),2))
print("Local disk total GiB:", round(gib(disk.total),2))
print("Local disk free GiB:", round(gib(disk.free),2))

print("\n=== CUDA ===")
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "This notebook requires a Colab GPU runtime."
print("GPU:", torch.cuda.get_device_name(0))
props=torch.cuda.get_device_properties(0)
print("VRAM GiB:", round(gib(props.total_memory),2))

# Use the host CPU efficiently for BLAS/image preprocessing without changing model numerics.
threads = max(1, min(cpu_count, 16))
os.environ["OMP_NUM_THREADS"] = str(threads)
os.environ["MKL_NUM_THREADS"] = str(threads)
torch.set_num_threads(threads)
try:
    torch.set_num_interop_threads(max(1, min(4, cpu_count//2)))
except RuntimeError:
    pass

print("\nCPU threads configured:", torch.get_num_threads())
print("\n=== NVIDIA-SMI ===")
subprocess.run(["nvidia-smi"], check=False)

In [ ]:
# Verify the immutable bundle, extract to fast local SSD, and verify the bundled exact child.
import tarfile, os, shutil, json

assert BUNDLE.exists(), f"Missing bundle: {BUNDLE}"
bundle_sha = sha256(BUNDLE)
print("Bundle SHA256:", bundle_sha)
assert bundle_sha == EXPECTED_BUNDLE_SHA, "Bundle SHA mismatch — stop."

if LOCAL_ROOT.exists():
    shutil.rmtree(LOCAL_ROOT)
LOCAL_ROOT.mkdir(parents=True)

print("Extracting bundle to local SSD...")
with tarfile.open(BUNDLE, "r:gz") as tf:
    tf.extractall(LOCAL_ROOT)

manifest_path = PAYLOAD / "BUNDLE_MANIFEST.json"
manifest = json.loads(manifest_path.read_text())
print(json.dumps(manifest, indent=2, sort_keys=True))

bundled_checkpoint = PAYLOAD / "state" / "child_v110_growth" / "infant_training.pt"
bundled_sha = sha256(bundled_checkpoint)
print("Bundled checkpoint SHA256:", bundled_sha)

assert manifest["state_checkpoint_sha256"] == EXPECTED_EXACT_EXPANSION_SHA
assert bundled_sha == EXPECTED_EXACT_EXPANSION_SHA
assert manifest["datasets"] == [
    "nursery_v090_bootstrap",
    "nursery_v095_mixed_003",
    "nursery_v096_detour_train",
    "nursery_v110_randomized_detour_train",
    "nursery_v110_two_barrier_train",
]

print("\nBundle and exact expansion checkpoint: VERIFIED")

In [ ]:
# Install the exact v0.11.0 project from the bundle.
import subprocess, sys, os
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(PROJECT), "--no-deps"], check=True)

os.chdir(PROJECT)
import continuous_brain, torch

print("Continuous Brain:", continuous_brain.__version__)
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

assert continuous_brain.__version__ == "0.11.0"

In [ ]:
# Initialize/synchronize the persistent Drive child, then copy the ACTIVE working state
# to local SSD. Training/evaluation operates locally; Drive is persistence.
import shutil, os, subprocess

DRIVE_STATE.parent.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
LOCAL_RESULTS.mkdir(parents=True, exist_ok=True)

if not (DRIVE_STATE / "infant_training.pt").exists():
    print("No persistent child found. Initializing from exact bundled expansion...")
    shutil.copytree(PAYLOAD / "state" / "child_v110_growth", DRIVE_STATE)
else:
    print("Persistent Drive child already exists; it will NOT be overwritten.")

if LOCAL_STATE.parent.exists():
    shutil.rmtree(LOCAL_STATE.parent)
LOCAL_STATE.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(DRIVE_STATE, LOCAL_STATE)

checkpoint = LOCAL_STATE / "infant_training.pt"
current_sha = sha256(checkpoint)
print("Active local checkpoint SHA256:", current_sha)

# Inspect resume metadata without altering the checkpoint.
saved = torch.load(checkpoint, map_location="cpu", weights_only=False)
metadata = dict(saved.get("metadata") or {})
route_steps = int(metadata.get("whole_brain_route_steps", 0))
print("whole_brain_route_steps:", route_steps)
print("growth_profile:", metadata.get("growth_profile"))

if route_steps == 0:
    assert current_sha == EXPECTED_EXACT_EXPANSION_SHA, "Route step 0 must be the exact expansion checkpoint."
elif route_steps == TARGET_ROUTE_STEPS:
    assert current_sha == EXPECTED_PHASE_A_SHA, (
        "A 6000-step checkpoint exists but its SHA differs from this lineage's completed Phase A checkpoint. "
        "Stop rather than silently mixing branches."
    )
elif route_steps > TARGET_ROUTE_STEPS:
    raise RuntimeError(f"Checkpoint is already beyond Phase A: {route_steps} > {TARGET_ROUTE_STEPS}")

print("Persistent/local state staging: OK")

In [ ]:
# Optional RAM-cache warmup.
# Reading the nursery tree once lets Linux use otherwise-idle host RAM as page cache,
# while keeping the authoritative data on local SSD.
if WARM_DATASETS_IN_RAM_CACHE:
    import time, os
    t0=time.time()
    total=0
    files=0
    for root, _, names in os.walk(DATA):
        for name in names:
            p=Path(root)/name
            try:
                with open(p,"rb", buffering=0) as f:
                    while True:
                        b=f.read(8*1024*1024)
                        if not b: break
                        total += len(b)
                files += 1
            except (OSError, PermissionError):
                pass
    print(f"Warmed {files:,} files / {total/1024**3:.2f} GiB into the OS page cache in {time.time()-t0:.1f}s.")
else:
    print("RAM-cache warmup disabled.")

In [ ]:
# Utilities: atomic Drive persistence, live CLI runner, and mixed-log -> clean-JSON extraction.
import json, shutil, subprocess, threading, time, re, os
from pathlib import Path

def atomic_copy(src: Path, dst: Path):
    src, dst = Path(src), Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + ".tmp-copy")
    shutil.copy2(src, tmp)
    os.replace(tmp, dst)

def sync_state_to_drive(include_snapshots=True):
    DRIVE_STATE.mkdir(parents=True, exist_ok=True)
    # Main checkpoint/metadata.
    for name in ("infant_training.pt","parents.json","identity.json"):
        p=LOCAL_STATE/name
        if p.exists():
            atomic_copy(p, DRIVE_STATE/name)
    # Immutable periodic snapshots.
    if include_snapshots:
        for p in sorted(LOCAL_STATE.glob("infant_training.v110_route_*.pt")):
            dst=DRIVE_STATE/p.name
            if not dst.exists() and time.time()-p.stat().st_mtime > 2:
                atomic_copy(p,dst)

def extract_last_json(text_or_path):
    if isinstance(text_or_path, (str,Path)) and Path(str(text_or_path)).exists():
        text=Path(str(text_or_path)).read_text(errors="replace")
    else:
        text=str(text_or_path)
    dec=json.JSONDecoder()
    best=None
    for i,ch in enumerate(text):
        if ch!="{": continue
        try:
            obj,end=dec.raw_decode(text[i:])
            if isinstance(obj,dict):
                best=obj
        except Exception:
            pass
    if best is None:
        raise ValueError("No JSON object found in log")
    return best

def run_streamed(cmd, log_path, cwd=PROJECT):
    log_path=Path(log_path)
    log_path.parent.mkdir(parents=True,exist_ok=True)
    print("$", " ".join(map(str,cmd)))
    with open(log_path,"w",buffering=1) as log:
        proc=subprocess.Popen(
            list(map(str,cmd)), cwd=str(cwd),
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1
        )
        for line in proc.stdout:
            print(line,end="")
            log.write(line)
            # Periodically persist completed snapshots without training directly on Drive.
            m=re.search(r"whole_brain_step=(\d+)", line)
            if m and int(m.group(1)) % 500 == 100:
                sync_state_to_drive(include_snapshots=True)
        rc=proc.wait()
    if rc != 0:
        raise RuntimeError(f"Command failed with exit code {rc}; see {log_path}")
    return log_path

In [ ]:
# Canonical Phase A resume/skip logic.
# IMPORTANT: --updates is ADDITIONAL updates, so this cell computes only the remainder.
saved = torch.load(LOCAL_STATE/"infant_training.pt", map_location="cpu", weights_only=False)
route_steps = int((saved.get("metadata") or {}).get("whole_brain_route_steps",0))
remaining = TARGET_ROUTE_STEPS - route_steps

if remaining == 0:
    print("Phase A already complete at 6000 routed steps — SKIPPING TRAINING.")
elif remaining < 0:
    raise RuntimeError("State is beyond the authorized Phase A boundary.")
elif not RUN_PHASE_A_IF_NEEDED:
    print(f"Phase A needs {remaining} additional updates, but RUN_PHASE_A_IF_NEEDED=False.")
else:
    print(f"Resuming canonical Phase A for exactly {remaining} additional updates.")
    cmd=[
        "python","-m","continuous_brain.parented_cli","train-whole-brain",
        "--rehearsal-datasets",
        str(DATA/"nursery_v090_bootstrap"),
        str(DATA/"nursery_v095_mixed_003"),
        "--world-datasets",
        str(DATA/"nursery_v095_mixed_003"),
        str(DATA/"nursery_v096_detour_train"),
        str(DATA/"nursery_v110_randomized_detour_train"),
        str(DATA/"nursery_v110_two_barrier_train"),
        "--planning-dataset", str(DATA/"nursery_v110_randomized_detour_train"),
        "--state-dir",str(LOCAL_STATE),
        "--updates",str(remaining),
        "--batch","8",
        "--learning-rate","0.0001",
        "--anchor-weight","0.10",
        "--checkpoint-every","500",
        "--seed","11010",
        "--device","cuda",
    ]
    log=run_streamed(cmd, LOCAL_RESULTS/"V110_WHOLE_BRAIN_GROWTH_PHASE_A.stdout.log")
    sync_state_to_drive(include_snapshots=True)
    atomic_copy(log, DRIVE_RESULTS/log.name)

# Re-check boundary.
saved = torch.load(LOCAL_STATE/"infant_training.pt", map_location="cpu", weights_only=False)
route_steps = int((saved.get("metadata") or {}).get("whole_brain_route_steps",0))
print("route_steps now:", route_steps)
assert route_steps == TARGET_ROUTE_STEPS

In [ ]:
# Freeze Phase A and produce a PURE JSON training result for the readiness checker.
# If your original Drive tee-log exists, sanitize it; otherwise use this run's local log.
phase_sha=sha256(LOCAL_STATE/"infant_training.pt")
print("Phase A checkpoint SHA256:", phase_sha)
assert phase_sha == EXPECTED_PHASE_A_SHA, "Completed Phase A SHA differs from the established lineage."

frozen = LOCAL_STATE/"infant_training.at_V110_PHASE_A_6000.pt"
if not frozen.exists():
    shutil.copy2(LOCAL_STATE/"infant_training.pt", frozen)
atomic_copy(frozen, DRIVE_STATE/frozen.name)
(DRIVE_STATE/(frozen.name+".sha256")).write_text(f"{phase_sha}  {frozen.name}\n")

candidate_logs = [
    DRIVE_RESULTS/"V110_WHOLE_BRAIN_GROWTH_PHASE_A.json",
    LOCAL_RESULTS/"V110_WHOLE_BRAIN_GROWTH_PHASE_A.stdout.log",
]
training_obj=None
for p in candidate_logs:
    if p.exists():
        try:
            obj=extract_last_json(p)
            if obj.get("format")=="continuous_brain.whole_brain_training.v110":
                training_obj=obj; break
        except Exception:
            pass

if training_obj is None:
    # Reconstruct the integrity fields only if the known canonical checkpoint hash matches.
    # This branch is not expected for your current run because the Drive tee-log exists.
    training_obj={
        "format":"continuous_brain.whole_brain_training.v110",
        "checkpoint":str(DRIVE_STATE/"infant_training.pt"),
        "checkpoint_sha256":phase_sha,
        "legacy_tensors_changed":0,
        "plasticity":"growth-only",
        "route_steps_before":0,
        "route_steps_after":6000,
        "route_counts":{"embodiment":600,"language":1200,"options":600,"perception":1200,"planning":1200,"world":1200},
        "schedule":["perception","language","world","embodiment","planning","perception","language","planning","world","options"],
        "evidence_origin":"reconstructed_from_known_canonical_phase_a_checkpoint_hash",
    }

assert training_obj["legacy_tensors_changed"]==0
assert training_obj["plasticity"]=="growth-only"
assert training_obj["route_steps_after"]==6000

training_clean=LOCAL_RESULTS/"V110_WHOLE_BRAIN_GROWTH_PHASE_A_CLEAN.json"
training_clean.write_text(json.dumps(training_obj,indent=2,sort_keys=True))
atomic_copy(training_clean, DRIVE_RESULTS/training_clean.name)
print(training_clean.read_text())

In [ ]:
# Reconstruct the clean expansion evidence from the immutable exact-expansion checkpoint
# if the original expansion JSON was not uploaded to Drive.
from continuous_brain.parented_train import load_checkpoint
from continuous_brain.growth_v110 import is_growth_parameter

exp_ckpt = PAYLOAD/"state"/"child_v110_growth"/"infant_training.pt"
assert sha256(exp_ckpt)==EXPECTED_EXACT_EXPANSION_SHA

model, exp_saved = load_checkpoint(exp_ckpt, "cpu")
emd=dict(exp_saved.get("metadata") or {})
audit=emd.get("growth_equivalence_audit") or {}

expanded_params=int(model.parameter_count)
growth_params=sum(
    p.numel() for n,p in model.named_parameters()
    if is_growth_parameter(n) and not n.startswith(("target_vision.","target_spatial."))
)
source_params=expanded_params-growth_params

assert audit.get("exact_function_preservation") is True
assert float(audit.get("max_abs_difference",1.0)) == 0.0
assert expanded_params >= 50_000_000
assert emd.get("growth_source_sha256") == "1936e046b21292a5d0519d03fe087f2568b3899e91e9bd6796a0b020a966b3a9"

expansion_obj={
    "format":"continuous_brain.function_preserving_growth.v110",
    "source_checkpoint_sha256":emd["growth_source_sha256"],
    "expanded_checkpoint_sha256":EXPECTED_EXACT_EXPANSION_SHA,
    "source_trainable_parameters":source_params,
    "expanded_trainable_parameters":expanded_params,
    "new_growth_parameters":growth_params,
    "legacy_tensor_mismatches":0,
    "new_growth_state_keys":len(emd.get("growth_new_state_keys") or []),
    "growth_profile":emd.get("growth_profile"),
    "equivalence":audit,
    "evidence_origin":"reconstructed_from_exact_canonical_checkpoint_and_saved_growth_audit",
}
exp_out=LOCAL_RESULTS/"V110_EXPANSION_CLEAN.json"
exp_out.write_text(json.dumps(expansion_obj,indent=2,sort_keys=True))
atomic_copy(exp_out, DRIVE_RESULTS/exp_out.name)
print(exp_out.read_text())

## Formal post-Phase-A qualification

The following cells use the canonical settings. They run from the **local SSD copy** of the frozen 6000-step state and write results back to Drive.

These tests are not all new blind evidence:

- bootstrap retention: regression;
- old P4b seeds 12001–12100: regression;
- v0.10.4 sealed world holdouts: regression;
- randomized-detour seeds 41001–41100: predeclared **development** test;
- seeds 51001–51100 remain untouched.

In [ ]:
# Gate 1 — bootstrap retention.
bootstrap_log=LOCAL_RESULTS/"V110_BOOTSTRAP_RETENTION.stdout.log"
cmd=[
    "python","-m","continuous_brain.parented_cli","evaluate",
    "--dataset",str(DATA/"nursery_v090_bootstrap"),
    "--state-dir",str(LOCAL_STATE),
    "--device","cuda",
    "--seed","17",
]
run_streamed(cmd,bootstrap_log)
bootstrap_obj=extract_last_json(bootstrap_log)
bootstrap_clean=LOCAL_RESULTS/"V110_BOOTSTRAP_RETENTION.json"
bootstrap_clean.write_text(json.dumps(bootstrap_obj,indent=2,sort_keys=True))
atomic_copy(bootstrap_log, DRIVE_RESULTS/bootstrap_log.name)
atomic_copy(bootstrap_clean, DRIVE_RESULTS/bootstrap_clean.name)

print("bootstrap passed:", bootstrap_obj.get("passed"))
print(json.dumps({
    "vision_similarity": bootstrap_obj.get("vision_similarity"),
    "language_similarity": bootstrap_obj.get("language_similarity"),
    "passed": bootstrap_obj.get("passed"),
}, indent=2))

In [ ]:
# Gate 2 — randomized-detour development evaluation.
generalized_clean=LOCAL_RESULTS/"V110_RANDOMIZED_DETOUR_DEV_41001_41100.json"
generalized_log=LOCAL_RESULTS/"V110_RANDOMIZED_DETOUR_DEV_41001_41100.stdout.log"

cmd=[
    "python","-m","continuous_brain.parented_cli","evaluate-generalized-planning",
    "--state-dir",str(LOCAL_STATE),
    "--kind","randomized_detour",
    "--episodes",str(FORMAL_RANDOMIZED_DEV_EPISODES),
    "--steps","160",
    "--seed-start",str(FORMAL_RANDOMIZED_DEV_SEED_START),
    "--option-depth","2",
    "--selector-topk","2",
    "--beam-width","8",
    "--max-option-steps","12",
    "--discount","0.95",
    "--min-commit","3",
    "--terminate-threshold","0.50",
    "--device","cuda",
    "--output",str(generalized_clean),
]
run_streamed(cmd,generalized_log)
atomic_copy(generalized_log, DRIVE_RESULTS/generalized_log.name)
atomic_copy(generalized_clean, DRIVE_RESULTS/generalized_clean.name)

g=json.loads(generalized_clean.read_text())
print("Randomized-detour development completion:", g["completion_rate"])
print("Growth-readiness threshold:", 0.20)
print("DEVELOPMENT GATE:", g["completion_rate"] >= 0.20)

In [ ]:
# Gate 3 — historical P4b hierarchical-planning regression.
p4_clean=LOCAL_RESULTS/"V110_P4B_REGRESSION_12001_12100.json"
p4_log=LOCAL_RESULTS/"V110_P4B_REGRESSION_12001_12100.stdout.log"

cmd=[
    "python","-m","continuous_brain.parented_cli","evaluate-hierarchical-options",
    "--state-dir",str(LOCAL_STATE),
    "--episodes",str(P4B_REGRESSION_EPISODES),
    "--steps","120",
    "--seed-start",str(P4B_REGRESSION_SEED_START),
    "--option-depth","2",
    "--selector-topk","2",
    "--beam-width","8",
    "--max-option-steps","12",
    "--discount","0.95",
    "--min-commit","3",
    "--terminate-threshold","0.50",
    "--device","cuda",
    "--output",str(p4_clean),
]
run_streamed(cmd,p4_log)
atomic_copy(p4_log, DRIVE_RESULTS/p4_log.name)
atomic_copy(p4_clean, DRIVE_RESULTS/p4_clean.name)

p4=json.loads(p4_clean.read_text())
print("P4b regression passed:",p4["p4b_passed"])
print("Hierarchical completion:",p4["hierarchical_option_planning"]["completion_rate"])
print("Reactive completion:",p4["reactive"]["completion_rate"])

## Gate 4 — v0.10.4 sealed world-retention regression

This requires one small archive containing the **original already-sealed v0.10.4 holdouts**.

If `v110_world_retention_assets.tar.gz` is not yet in `MyDrive/continuous_brain_v110/`, run the provided `prepare_v110_world_retention_assets.sh` once in WSL and upload the resulting `.tar.gz` and `.sha256` files.

The notebook verifies:

1. the original seal SHA (`8f00579c...`);
2. the original foraging transition digest (`1797991c...`);
3. the original detour transition digest (`a0ba1cb4...`);
4. the copied datasets' computed digests.

Only after all four checks pass does it write a **relocation-only** seal copy whose holdout paths point to `/content`. The original seal is retained unchanged.

In [ ]:
# Prepare and verify the relocated world-retention assets.
WORLD_READY=False
world_root=Path("/content/v110_world_retention")
relocated_seal=LOCAL_RESULTS/"V104_WORLD_HOLDOUT_SEAL_COLAB_RELOCATED.json"

if not WORLD_ASSETS.exists():
    print("WORLD RETENTION ASSET ARCHIVE IS NOT YET IN DRIVE.")
    print("Expected:", WORLD_ASSETS)
    print("\nRun prepare_v110_world_retention_assets.sh in WSL, then upload the archive and .sha256.")
else:
    if WORLD_ASSETS_SHA.exists():
        expected_line=WORLD_ASSETS_SHA.read_text().strip().split()[0]
        actual=sha256(WORLD_ASSETS)
        print("World-assets archive SHA:",actual)
        assert actual==expected_line, "World-assets archive SHA mismatch"

    if world_root.exists():
        shutil.rmtree(world_root)
    world_root.mkdir(parents=True)
    with tarfile.open(WORLD_ASSETS,"r:gz") as tf:
        tf.extractall(world_root)

    original_seal=world_root/"original_seal.json"
    assert original_seal.exists()
    original_seal_sha=sha256(original_seal)
    print("Original v0.10.4 seal SHA:",original_seal_sha)
    assert original_seal_sha==EXPECTED_V104_SEAL_SHA

    from continuous_brain.world_consolidation_v104 import transition_digest
    foraging=world_root/"foraging"
    detour=world_root/"detour"
    fd=transition_digest(foraging)
    dd=transition_digest(detour)
    print("Foraging digest:",fd)
    print("Detour digest:  ",dd)
    assert fd==EXPECTED_FORAGING_HOLDOUT_DIGEST
    assert dd==EXPECTED_DETOUR_HOLDOUT_DIGEST

    seal=json.loads(original_seal.read_text())
    assert seal["holdouts"]["foraging"]["transition_digest"]==fd
    assert seal["holdouts"]["detour"]["transition_digest"]==dd

    relocated=json.loads(original_seal.read_text())
    relocated["holdouts"]["foraging"]["path"]=str(foraging.resolve())
    relocated["holdouts"]["detour"]["path"]=str(detour.resolve())
    relocated["colab_relocation"]={
        "original_seal_sha256":original_seal_sha,
        "only_paths_changed":True,
        "foraging_transition_digest":fd,
        "detour_transition_digest":dd,
        "purpose":"Regression-only relocation of byte-identical sealed holdouts; not a new blind seal.",
    }
    relocated_seal.write_text(json.dumps(relocated,indent=2,sort_keys=True))
    atomic_copy(relocated_seal,DRIVE_RESULTS/relocated_seal.name)
    WORLD_READY=True
    print("\nWorld-retention assets: VERIFIED AND READY")

In [ ]:
# Run Gate 4 if the assets are ready.
world_clean=LOCAL_RESULTS/"V110_WORLD_RETENTION.json"
world_log=LOCAL_RESULTS/"V110_WORLD_RETENTION.stdout.log"

if not WORLD_READY:
    print("Skipping world-retention evaluation until the asset archive is uploaded.")
else:
    cmd=[
        "python","-m","continuous_brain.parented_cli","evaluate-world-consolidation",
        "--state-dir",str(LOCAL_STATE),
        "--foraging-holdout",str(world_root/"foraging"),
        "--detour-holdout",str(world_root/"detour"),
        "--holdout-seal",str(relocated_seal),
        "--device","cuda",
        "--output",str(world_clean),
    ]
    run_streamed(cmd,world_log)
    atomic_copy(world_log,DRIVE_RESULTS/world_log.name)
    atomic_copy(world_clean,DRIVE_RESULTS/world_clean.name)
    w=json.loads(world_clean.read_text())
    print("World consolidation regression passed:",w["world_consolidation_passed"])
    print("Foraging:",w["domains"]["foraging"]["world_model_passed"])
    print("Detour:",w["domains"]["detour"]["world_model_passed"])

In [ ]:
# Final v0.11.0 growth-readiness gate.
# It runs automatically once all required clean result files exist.
required = {
    "expansion": LOCAL_RESULTS/"V110_EXPANSION_CLEAN.json",
    "training": LOCAL_RESULTS/"V110_WHOLE_BRAIN_GROWTH_PHASE_A_CLEAN.json",
    "bootstrap": LOCAL_RESULTS/"V110_BOOTSTRAP_RETENTION.json",
    "world": LOCAL_RESULTS/"V110_WORLD_RETENTION.json",
    "p4b": LOCAL_RESULTS/"V110_P4B_REGRESSION_12001_12100.json",
    "generalized": LOCAL_RESULTS/"V110_RANDOMIZED_DETOUR_DEV_41001_41100.json",
}
missing=[str(p) for p in required.values() if not p.exists()]

if missing:
    print("Readiness gate is pending. Missing:")
    for p in missing: print(" -",p)
else:
    readiness_log=LOCAL_RESULTS/"V110_GROWTH_READINESS.stdout.log"
    cmd=[
        "python",str(PROJECT/"scripts"/"check_v110_growth_readiness.py"),
        "--expansion-result",str(required["expansion"]),
        "--training-result",str(required["training"]),
        "--bootstrap-result",str(required["bootstrap"]),
        "--world-retention-result",str(required["world"]),
        "--p4b-regression-result",str(required["p4b"]),
        "--generalized-result",str(required["generalized"]),
    ]
    run_streamed(cmd,readiness_log)
    readiness=extract_last_json(readiness_log)
    readiness_clean=LOCAL_RESULTS/"V110_GROWTH_READINESS.json"
    readiness_clean.write_text(json.dumps(readiness,indent=2,sort_keys=True))
    atomic_copy(readiness_log,DRIVE_RESULTS/readiness_log.name)
    atomic_copy(readiness_clean,DRIVE_RESULTS/readiness_clean.name)

    print("\n=== V0.11.0 GROWTH READINESS ===")
    for k,v in readiness["gates"].items():
        print(f"{'PASS' if v else 'FAIL':4s}  {k}")
    print("\ngrowth_phase_passed:",readiness["growth_phase_passed"])
    print(readiness["meaning"])

In [ ]:
# Produce a compact durable status report and resource snapshot.
import datetime, subprocess

status={
    "timestamp_utc":datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "continuous_brain_version":"0.11.0",
    "gpu":torch.cuda.get_device_name(0),
    "torch":torch.__version__,
    "phase_a_checkpoint_sha256":sha256(LOCAL_STATE/"infant_training.pt"),
    "phase_a_route_steps":int((torch.load(LOCAL_STATE/"infant_training.pt",map_location="cpu",weights_only=False).get("metadata") or {}).get("whole_brain_route_steps",0)),
    "reserved_blind_seeds_untouched":"51001-51100",
    "results_present":{p.name:p.exists() for p in required.values()},
}
ready_path=LOCAL_RESULTS/"V110_COLAB_STATUS.json"
ready_path.write_text(json.dumps(status,indent=2,sort_keys=True))
atomic_copy(ready_path,DRIVE_RESULTS/ready_path.name)

print(ready_path.read_text())
print("\n=== GPU MEMORY ===")
subprocess.run([
    "nvidia-smi",
    "--query-gpu=name,memory.used,memory.total,utilization.gpu,temperature.gpu,power.draw",
    "--format=csv"
],check=False)
print("\nAll durable results are under:",DRIVE_RESULTS)

## Optional resource-saturation work — **not part of the formal gate**

The canonical v0.11.0 experiment uses batch 8 and the planning evaluations are inherently sequential. A 53.68M model therefore cannot responsibly consume all 40 GB of A100 VRAM without changing the experiment.

After the formal gates are complete, a **disposable throughput benchmark** can test larger batches for a future version/phase. Do not use that benchmark to rewrite the already-completed Phase A result, and do not touch the reserved blind seeds `51001–51100`.